<a href="https://colab.research.google.com/github/raiissha29/Naan-Mudhalvan-Project/blob/main/nm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q --upgrade pip
!pip install -q gradio==4.29.0 huggingface_hub==0.24.6 json5==0.9.25 jsonschema==4.21.1 pydantic==2.7.1 python-dotenv==1.0.1 typing-extensions==4.11.0 colorama==0.4.6 tabulate==0.9.0 openpyxl==3.1.2 pandas==2.2.0

In [2]:
import json
import json5
import pandas as pd
import gradio as gr
import re
from enum import Enum
from jsonschema import validate, ValidationError

In [3]:
class CaseStyle(Enum):
    SNAKE_CASE = "snake_case"
    CAMEL_CASE = "camelCase"
    KEBAB_CASE = "kebab-case"
    PASCAL_CASE = "PascalCase"

In [4]:
class JSONRefinerCore:

    def __init__(self):
        self.history=[]

core=JSONRefinerCore()

In [5]:
def infer_type(value):

    v=str(value).strip().lower()

    if v in ["true","yes","on"]:
        return True

    if v in ["false","no","off"]:
        return False

    if v in ["null","none","nil","undefined","n/a"]:
        return None

    try:
        if "." in v:
            return float(v)
        return int(v)
    except:
        pass

    return value

In [6]:
def convert_case_key(key,style):

    words=re.split(r'[\s_\-]+',key)

    if style=="snake":
        return "_".join(w.lower() for w in words)

    if style=="kebab":
        return "-".join(w.lower() for w in words)

    if style=="camel":
        return words[0].lower()+"".join(w.title() for w in words[1:])

    if style=="pascal":
        return "".join(w.title() for w in words)

    return key


def case_convert_json(data,style):

    obj=json.loads(data)

    result={}

    for k,v in obj.items():
        result[convert_case_key(k,style)]=v

    core.history.append(f"🔤 Case Conversion : Converted {len(result)} keys to {style} case")

    return json.dumps(result,indent=4)

In [7]:
def text_to_json(text):

    result={}

    for line in text.split("\n"):

        if ":" in line:
            k,v=line.split(":",1)

            result[k.strip()]=infer_type(v.strip())

    core.history.append(f"🔄 Text → JSON : Converted {len(result)} fields → {list(result.keys())}")

    return json.dumps(result,indent=4)

In [8]:
def json_statistics(data):

    obj=json.loads(data)

    stats={
        "total_keys":len(obj),
        "data_types":list(set(type(v).__name__ for v in obj.values()))
    }

    core.history.append(f"📊 JSON Statistics : {len(obj)} keys, types → {stats['data_types']}")

    return json.dumps(stats,indent=4)

In [9]:
# CELL 9 — Flatten & Unflatten

def flatten_json(obj, parent="", sep="."):

    items = {}

    for k, v in obj.items():

        new_key = f"{parent}{sep}{k}" if parent else k

        if isinstance(v, dict):
            items.update(flatten_json(v, new_key, sep))

        else:
            items[new_key] = v

    return items


def flatten_run(data):

    obj = json.loads(data)

    flat = flatten_json(obj)

    core.history.append(f"Flatten JSON : {len(flat)} flattened keys generated")

    return json.dumps(flat, indent=4)


def unflatten_json(d):

    result = {}

    for k, v in d.items():

        parts = k.split(".")

        target = result

        for part in parts[:-1]:
            target = target.setdefault(part, {})

        target[parts[-1]] = v

    return result


def unflatten_run(data):

    obj = json.loads(data)

    core.history.append(f"Unflatten JSON : Restored nested structure from {len(obj)} flattened keys")

    return json.dumps(unflatten_json(obj), indent=4)

In [10]:
# CELL 10 — JSON Validation

def validate_json(data, schema=None):

    try:
        obj = json.loads(data)

        # If schema is provided → perform schema validation
        if schema and schema.strip():

            schema_obj = json.loads(schema)

            validate(instance=obj, schema=schema_obj)

            core.history.append("Validation : JSON is valid and matches schema")

            return "JSON is VALID and matches the schema"

        # Only syntax validation
        core.history.append("Validation : JSON syntax verified")

        return "JSON syntax is VALID"

    except ValidationError as e:

        # Extract field name if possible
        field = ""
        if e.path:
            field = list(e.path)[0]

        expected = ""
        if isinstance(e.schema, dict) and "type" in e.schema:
            expected = e.schema["type"]

        if field and expected:
            msg = f"Schema Error : {field} must be {expected}"
        else:
            msg = f"Schema Validation Error : {e.message}"

        core.history.append(f"Validation Failed : {msg}")

        return msg

    except json.JSONDecodeError as e:

        core.history.append(f"Invalid JSON syntax at line {e.lineno}")

        return f"JSON Syntax Error at line {e.lineno}, column {e.colno}"

    except Exception as e:

        core.history.append("Unknown validation error")

        return f"Validation Error : {str(e)}"

In [11]:
def highlight_error(data):

    try:
        json.loads(data)
        return "No errors"

    except json.JSONDecodeError as e:
        return f"Error at line {e.lineno}, column {e.colno}: {e.msg}"

In [12]:
def merge_json(j1,j2,j3):

    result={}

    for j in [j1,j2,j3]:

        if j.strip():
            result.update(json.loads(j))

    core.history.append(f"🔗 Merge JSON : Combined inputs into {len(result)} total keys")

    return json.dumps(result,indent=4)

In [13]:
def generate_schema(data):

    obj=json.loads(data)

    schema={"type":"object","properties":{}}

    for k,v in obj.items():
        schema["properties"][k]={"type":type(v).__name__}

    core.history.append(f"📑 Schema Generator : Schema created for keys {list(obj.keys())}")

    return json.dumps(schema,indent=4)

In [14]:
def export_excel(data):

    obj=json.loads(data)

    df=pd.json_normalize(obj)

    file1="output.xlsx"
    file2="output.csv"

    df.to_excel(file1,index=False)
    df.to_csv(file2,index=False)

    core.history.append(f"📁 Export : JSON exported with {len(obj)} fields → Excel & CSV")

    return f"Files created: {file1}, {file2}"

In [15]:
 def view_history():
    if not core.history:
        return "No history available"
    return "\n".join(core.history)


def clear_history():
    core.history.clear()
    return "History Cleared"


def user_guide():
    return """
===============================
JSON REFINER ADVANCED EDITION
User Guide
===============================

This application is a Smart JSON Processing Toolkit designed to help users
convert, analyze, validate, and transform JSON data efficiently.

The system is divided into several modules, each performing a specific
operation on JSON data.

----------------------------------------------------
1. CORE REFINING MODULE
----------------------------------------------------

This module performs basic JSON transformation and analysis operations.

1.1 Convert Text → JSON
This feature converts simple key-value text input into structured JSON format.

Example Input:
name: John
age: 22
city: Chennai

Output JSON:
{
    "name": "John",
    "age": 22,
    "city": "Chennai"
}

The system automatically detects data types such as:
- Integers
- Floats
- Boolean values
- Null values

This helps convert raw text data into machine-readable JSON format.

----------------------------------------------------

1.2 Flatten JSON
Flattening converts nested JSON objects into a single-level structure.

Example Input:
{
  "name": "John",
  "address": {
        "city": "Chennai",
        "pincode": 600001
  }
}

Flattened Output:
{
  "name": "John",
  "address.city": "Chennai",
  "address.pincode": 600001
}

Flattening is useful for:
• Data analysis
• Database storage
• Spreadsheet processing

----------------------------------------------------

1.3 Unflatten JSON
Unflattening restores flattened JSON back into nested structure.

Example Input:
{
 "address.city": "Chennai",
 "address.pincode": 600001
}

Output:
{
 "address":{
    "city":"Chennai",
    "pincode":600001
 }
}

This allows reconstruction of hierarchical JSON data.

----------------------------------------------------

1.4 Case Conversion
This feature converts JSON keys into different naming styles.

Supported Case Styles:

snake_case
example_key_name

camelCase
exampleKeyName

kebab-case
example-key-name

PascalCase
ExampleKeyName

This is useful when JSON needs to match API or programming conventions.

----------------------------------------------------

1.5 JSON Statistics
This module analyzes JSON data and generates useful information.

Statistics include:
• Total number of keys
• Types of values used in JSON

Example Output:
{
 "total_keys": 4,
 "data_types": ["str","int"]
}

This helps developers understand the structure of the JSON data.

----------------------------------------------------
2. JSON VALIDATION MODULE
----------------------------------------------------

This module verifies whether JSON data is correct.

2.1 JSON Syntax Validation
Checks if the JSON structure is valid according to JSON standards.

Example error detection:
• Missing commas
• Incorrect brackets
• Invalid quotes

The system reports the exact line and column of the error.

----------------------------------------------------

2.2 JSON Schema Validation
Users can provide a JSON schema to validate the structure of JSON data.

Example Schema:
{
 "type": "object",
 "properties": {
    "name": {"type": "string"},
    "age": {"type": "integer"}
 }
}

The system checks whether the JSON data follows the defined schema rules.

----------------------------------------------------

2.3 Error Highlighting
If JSON contains syntax errors, the system highlights the exact
location of the error including:

• Line number
• Column number
• Description of the issue

This helps users quickly identify and fix JSON problems.

----------------------------------------------------
3. MERGE JSON MODULE
----------------------------------------------------

This module combines multiple JSON inputs into a single JSON object.

Users can provide up to three JSON inputs.

Example:

JSON 1:
{"name":"John"}

JSON 2:
{"age":25}

Merged Output:
{
 "name":"John",
 "age":25
}

This is useful for:
• Combining data from different sources
• Data integration
• API response aggregation

----------------------------------------------------
4. SCHEMA GENERATION AND EXPORT MODULE
----------------------------------------------------

4.1 JSON Schema Generator

This feature automatically generates a JSON schema based on
the provided JSON data.

Example Input:
{
 "name":"John",
 "age":25
}

Generated Schema:
{
 "type":"object",
 "properties":{
   "name":{"type":"string"},
   "age":{"type":"integer"}
 }
}

This helps developers quickly generate schema definitions.

----------------------------------------------------

4.2 Export JSON to Excel / CSV

This feature converts JSON data into spreadsheet formats.

Supported formats:
• Excel (.xlsx)
• CSV (.csv)

Benefits:
• Easy data analysis
• Spreadsheet compatibility
• Data sharing

----------------------------------------------------
5. HISTORY AND GUIDE MODULE
----------------------------------------------------

5.1 View History
Displays all previously performed operations within the system.

Examples of logged actions:
• Text converted to JSON
• JSON flattened
• Case conversion performed
• JSON validated
• Files exported

This helps track user activity and processing steps.

----------------------------------------------------

5.2 Delete History
Clears all stored operation history from the system.

This allows users to reset the activity log.

----------------------------------------------------

5.3 User Guide
Displays this complete documentation explaining
all modules and their functions.

----------------------------------------------------

SYSTEM BENEFITS
----------------------------------------------------

The JSON Refiner Advanced Edition provides:

• Automated JSON processing
• Error detection and debugging
• Data transformation utilities
• Schema generation
• Export capabilities
• User activity tracking

This tool simplifies JSON handling for developers,
data analysts, and system administrators.

===============================
End of User Guide
===============================
"""

In [16]:
with gr.Blocks(
    title="JSON Refiner Advanced Edition",
    theme=gr.themes.Soft(),
    css="footer{display:none !important}"
) as app:

    gr.Markdown(
"""
# 🧠 JSON Refiner Advanced Edition
### 🚀 Smart JSON Processing Toolkit

✨ Convert • Validate • Merge • Analyze • Export JSON Data
"""
)

    with gr.Tabs():

        # ⚙️ Core Refining
        with gr.Tab("⚙️ Core Refining"):

            txt=gr.Textbox(lines=6,label="📝 Text Input")
            btn=gr.Button("🔄 Convert Text → JSON")
            out=gr.Code()

            btn.click(text_to_json,txt,out)

            j=gr.Textbox(lines=8,label="📄 JSON Input")

            flatten_btn=gr.Button("📂 Flatten JSON")
            unflatten_btn=gr.Button("📑 Unflatten JSON")
            stats_btn=gr.Button("📊 JSON Statistics")

            out2=gr.Code()

            flatten_btn.click(flatten_run,j,out2)
            unflatten_btn.click(unflatten_run,j,out2)
            stats_btn.click(json_statistics,j,out2)

            gr.Markdown("### 🔤 Case Conversion")

            case_input=gr.Textbox(lines=8,label="📄 JSON for Case Conversion")

            style=gr.Dropdown(
                ["snake","camel","kebab","pascal"],
                label="🎨 Select Case Style",
                value="snake"
            )

            case_btn=gr.Button("🔁 Convert Case")

            case_out=gr.Code()

            case_btn.click(case_convert_json,[case_input,style],case_out)

        # ✅ Validation
        with gr.Tab("✅ Validation"):

            j=gr.Textbox(lines=8,label="📄 JSON Input")

            schema=gr.Textbox(
                lines=8,
                label="📑 JSON Schema (Optional)",
                placeholder="Paste JSON schema here if you want schema validation"
            )

            validate_btn=gr.Button("✔️ Validate JSON")
            error_btn=gr.Button("🚨 Highlight Error")

            out=gr.Textbox()

            # Validation (syntax + schema if provided)
            validate_btn.click(validate_json,[j,schema],out)

            # Error highlight
            error_btn.click(highlight_error,j,out)

        # 🔗 Merge JSON
        with gr.Tab("🔗 Merge JSON"):

            j1=gr.Textbox(lines=4,label="📄 JSON 1")
            j2=gr.Textbox(lines=4,label="📄 JSON 2")
            j3=gr.Textbox(lines=4,label="📄 JSON 3")

            btn=gr.Button("🔀 Merge JSON")

            out=gr.Code()

            btn.click(merge_json,[j1,j2,j3],out)

        # 📦 Schema + Export
        with gr.Tab("📦 Schema + Export"):

            j=gr.Textbox(lines=8,label="📄 JSON Input")

            schema_btn=gr.Button("🧩 Generate Schema")
            export_btn=gr.Button("📤 Export Excel / CSV")

            out=gr.Code()
            file=gr.Textbox(label="📁 File Path")

            schema_btn.click(generate_schema,j,out)
            export_btn.click(export_excel,j,file)

        # 📜 History + Guide
        with gr.Tab("📜 History + Guide"):

            view_btn=gr.Button("📖 View History")
            clear_btn=gr.Button("🗑️ Delete History")
            guide_btn=gr.Button("📘 User Guide")

            hist=gr.Textbox(lines=10)

            view_btn.click(view_history,None,hist)
            clear_btn.click(clear_history,None,hist)
            guide_btn.click(user_guide,None,hist)

app.launch()

IMPORTANT: You are using gradio version 4.29.0, however version 4.44.1 is available, please upgrade.
--------
Setting queue=True in a Colab notebook requires sharing enabled. Setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Running on public URL: https://ea59bff834b326a194.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from Terminal to deploy to Spaces (https://huggingface.co/spaces)


In [17]:
app.launch(share=True)

Rerunning server... use `close()` to stop if you need to change `launch()` parameters.
----
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Running on public URL: https://ea59bff834b326a194.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from Terminal to deploy to Spaces (https://huggingface.co/spaces)
